[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.0 MB/s eta 0:00:00


In [2]:
import torch
import math

In [11]:
# ✏️ YOUR IMPLEMENTATION HERE

def rope_implementation(x, base=10000):
    B, N, D = x.shape
    assert D % 2 == 0
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]
    positions = torch.arange(N, dtype=torch.float32)
    theta = torch.arange(D//2)
    frequencies = 1.0 / (base ** (2 * theta / D))
    outer_product = torch.outer(positions, frequencies)
    cos_cache = torch.cos(outer_product)
    sin_cache = torch.sin(outer_product)
    cos = cos_cache[:N].view(1, N, -1)
    sin = sin_cache[:N].view(1, N, -1)
    x_even_rot = x_even * cos - x_odd * sin
    x_odd_rot = x_even * sin + x_odd * cos
    result = torch.empty_like(x)
    result[..., 0::2] = x_even_rot
    result[..., 1::2] = x_odd_rot
    return result

def apply_rope(q, k):
    # 1. Compute position angles
    # 2. Split into even/odd pairs
    # 3. Apply rotation
    return rope_implementation(q), rope_implementation(k)

In [12]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print('Shape preserved:', qr.shape == q.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

Shape preserved: True
Norm preserved: True


In [13]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shapes (2.2ms)
  ✅ [2/4] Preserves norm (2.7ms)
  ✅ [3/4] Relative position property (2.6ms)
  ✅ [4/4] Gradient flow (1.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (8.7ms total)
  Progress saved. Run status() to see your dashboard.

